# Writing Spark configurations

- Now that you've reviewed some of the Spark configurations on your cluster, you want to modify some of the settings to tune Spark to your needs. You'll import some data to review that your changes have affected the cluster.

- The spark configuration is initially set to the default value of 200 partitions.

- The `spark` object is available for use. A file named `departures.txt.gz` is available for import. An initial DataFrame containing the distinct rows from departures.txt.gz is available as `departures_df`.

## Instructions

- Store the number of partitions in `departures_df` in the variable before.
- Change the `spark.sql.shuffle.partitions` configuration to 500 partitions.
- Recreate the `departures_df` DataFrame reading the distinct rows from the departures file.
- Print the number of partitions from before and after the configuration change.

In [1]:
# Explicitly Point Spark to the Correct Python Executable
# If you already have numpy installed in a specific environment (like conda or a virtualenv), but PySpark is defaulting to /usr/bin/python3 or system Python, 
# explicitly set the Python environment variables before initializing SparkContext:

import os
import sys

# Set these before initializing SparkContext (sc)
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
from pyspark.sql import SparkSession

In [3]:
# If you are developing an application you should use local mode. 
# Running in yarn mode is bit time consuming than running in local mode
# If you still insist to run in yarn mode then make sure that
# 1. You assigned more than one processor in the VM settings
# 2. Run only one kernel at a time

In [4]:
# spark = SparkSession. \
#     builder. \
#     enableHiveSupport(). \
#     appName('Demo'). \
#     master('yarn'). \
#     getOrCreate()

In [5]:
spark = SparkSession. \
    builder. \
    enableHiveSupport(). \
    appName('Demo'). \
    master('local[*]'). \
    getOrCreate()

In [6]:
sc = spark.sparkContext

In [9]:
departures_df = spark.read.csv('file:///home/talentum/UBUNTU_HOME/test-jupyter/P3/M3/SM3/3_clusterconfigurations/Dataset/departures.txt.gz').distinct()
departures_df = departures_df.repartition(200)

# Store the number of partitions in variable
before = departures_df.rdd.getNumPartitions()

# Configure Spark to use 500 partitions
spark.conf.set('spark.sql.shuffle.partitions', '500')

# Recreate the DataFrame using the departures data file
departures_df = spark.read.csv('file:///home/talentum/UBUNTU_HOME/test-jupyter/P3/M3/SM3/3_clusterconfigurations/Dataset/departures.txt.gz').distinct()

# Print the number of partitions for each instance
print("Partition count before change: %d" % before)
print("Partition count after change: %d" % departures_df.rdd.getNumPartitions())


Partition count before change: 200
Partition count after change: 500


In [10]:
departures_df.explain()

== Physical Plan ==
*(2) HashAggregate(keys=[_c0#156, _c1#157, _c2#158, _c3#159], functions=[])
+- Exchange hashpartitioning(_c0#156, _c1#157, _c2#158, _c3#159, 500), ENSURE_REQUIREMENTS, [id=#278]
   +- *(1) HashAggregate(keys=[_c0#156, _c1#157, _c2#158, _c3#159], functions=[])
      +- FileScan csv [_c0#156,_c1#157,_c2#158,_c3#159] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex[file:/home/talentum/UBUNTU_HOME/test-jupyter/P3/M3/SM3/3_clusterconfigurations/..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<_c0:string,_c1:string,_c2:string,_c3:string>


